# Declared Analysis Protocol

**Milestone M7 · studio notebook**

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/honr_464_logo.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/ms07_declared_analysis_protocol_student.ipynb)

---

This is your **Friday-studio workbench** for Milestone M7. The topic this week was
prediction, and this milestone turns it into your own project's spine: a written
statement of exactly how you will analyze your data, fixed **before you touch it**,
so no result can be reverse-engineered into a flattering method. The notebook gives
you a scaffold to flag any feature that would leak, a prompt to widen your
leakage-suspect search, a prompt to attack your own claim boundary, and the studio
red-team protocol you run with a partner and the Prediction & Leakage Auditor. The
full brief, rubric, and due date live on Brightspace under **M07**.

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

SEED = 464  # course number — keeps everything in this notebook reproducible
rng = np.random.default_rng(SEED)

print("✓ Setup complete — seed", SEED)

## 🎯 Definition of Done

You are done with this sprint when your submission contains all seven parts of the
deliverable, applied to **your own project**:

1. **The prediction verdict.** Either the four-part **contract** in order (target,
   baseline, split, metric) if your project forecasts unseen cases, or a defended
   one-sentence reason prediction is the wrong question, naming the question your
   project really asks and the design that answers it.
2. **The leakage trace.** The one feature whose value could be settled at or after
   your outcome, plus the **timing check** and **correlation check** that rule it
   out. Timing, not accuracy, decides.
3. **The honesty checks and retract triggers.** Which of the five checks apply
   (overfitting, cross-validation, calibration, subgroup performance, distribution
   shift), and for each, the result that would make you retract. Plus your
   **model-selection rule** fixed before the scores, and your criteria for calling
   the analysis **not useful**.
4. **The claim boundary.** One sentence naming the exact finding your protocol
   licenses, carrying its uncertainty, and the crossing it forbids: no causal
   reading of any weight.
5. **The Prediction & Leakage Auditor critique with your fixes.** Its leakage trace
   and held-out/baseline check pasted, each flag answered with your written fix, and
   the **one recompute** you would run yourself to check any AI or collaborator.
6. **AI Research Ledger rows.** One row for every AI-assisted step, each with a
   named verification method. A missing ledger returns the submission.
7. **The dossier update line** naming where each part now lives in your Research
   Project Dossier.

You present a 3-minute protocol walkthrough at the studio, and a partner plus the
auditor cross-review it. You revise from what survives.

> **A question that often comes up here:** *"What if my honest verdict is that my
> project does not predict?"* Then the defended non-prediction verdict **is** your
> deliverable, and writing it well earns full credit. You still name the too-good-to-
> be-true trap nearest your approach, a measure that quietly contains its own
> outcome, and your check for it. The defect this milestone catches is not "my
> project is descriptive." It is reporting a score with no baseline, or reading a
> cause off a model's weights.

### 🤝 AI Research Partner

AI is your arm and your research assistant in this sprint, not your brain. Hand it
the mechanical work; keep every judgment for yourself.

**Good to delegate here:**
- Widening your list of *candidate* leakage suspects, so you have a longer set to
  check against your own timeline.
- Explaining or extending the scaffold code below.
- Playing hostile reviewer against a claim boundary, or a contract, **you** already
  wrote.

**Never delegate (these are yours to decide):**
- The target you are predicting and the baseline you must beat.
- Whether a feature would actually exist at your prediction moment. A tool that has
  never seen your data's timeline can only guess.
- The verdict on whether your project should predict at all, and the one sentence
  your evidence licenses. A boundary you did not reason to is one you cannot defend
  at the studio.

Whatever you delegate, verify what comes back against the numbers your own cell
printed, and log it in your **AI Research Ledger** below. Code that runs without an
error is not the same as code that computed the right thing. See
`ai_resources/human_responsibility_checklist.md` for the full never-delegate list.

In [ ]:
# Declared-protocol scaffold — EDIT the values to match YOUR project.
# It does two edit-safe jobs, and both are reproducible under SEED = 464:
#   (1) flags any candidate feature that would LEAK (settled at/after the outcome),
#   (2) shows the baseline your model must beat for free, at your own base rate.
# Replace every "EDIT ME" with your project's words, then rerun.

# --- Part 1: leakage-timing audit ------------------------------------------
TARGET = "EDIT ME: the one column you are predicting"
# (feature name, when its value is settled relative to the PREDICTION MOMENT):
#   "before" = known before you forecast, safe to use
#   "at" / "after" = settled at or after the outcome, a LEAK
candidate_features = [
    ("EDIT ME: feature 1", "before"),
    ("EDIT ME: feature 2", "after"),
    ("EDIT ME: feature 3", "before"),
]

audit = pd.DataFrame(candidate_features, columns=["feature", "settled"])
audit["verdict"] = audit["settled"].map({
    "before": "safe: known at prediction time",
    "at":     "LEAK: settled at the outcome",
    "after":  "LEAK: settled after the outcome",
})
print(f"TARGET = {TARGET}")
print(audit.to_string(index=False))
leaks = audit.loc[audit["settled"] != "before", "feature"].tolist()
print(f"\n{len(leaks)} feature(s) to drop or re-time before modeling:", leaks or "none")

# --- Part 2: the free baseline ---------------------------------------------
BASE_RATE = 0.60   # EDIT ME — the share of your most common outcome, as 0..1
# Draw a reproducible held-out set at that base rate. The majority-class rule
# scores the base rate itself, so this is the bar your model has to clear.
holdout = rng.random(1000) < BASE_RATE
baseline_accuracy = max(holdout.mean(), 1 - holdout.mean())
print(f"\nAt a base rate of {BASE_RATE:.0%}, the majority-class baseline scores "
      f"about {baseline_accuracy:.1%} for free.")
print("Your model has to beat THAT, on held-out cases, to have earned anything.")

> 💡 **Gemini Prompt:** "I am declaring a prediction analysis protocol. Here
> is my target and the moment I would need the forecast: [your outcome, and when it
> is decided]. Here is my candidate feature list with when each value is recorded:
> [your list]. For each feature, tell me whether its value is settled before, at, or
> after the outcome, and flag any that could not exist at prediction time. Do not
> pick my features for me."
>
> **After running, verify (counters *fabricated plausibility*: a fluent list can
> flag a feature that is fine in your data and clear one that leaks):**
> - [ ] Check every flag against your *actual* data timeline. The timing check, not
>   the tool, decides. Re-derive the timing for any feature it clears.
> - [ ] Keep the single feature you most suspect at the top of your leakage trace,
>   even if the tool ranked it low. You know your outcome window; it is guessing.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

> 💡 **Gemini Prompt:** "Here is the one-sentence claim boundary I wrote for
> my analysis protocol: '[paste it].' My baseline scores [number] and my model scores
> [number] on held-out cases. Act as a hostile peer reviewer. Name every word that
> claims more than a held-out margin over that baseline can support, and flag any
> phrase that reads a cause off the model. Do not rewrite the sentence for me."
>
> **After running, verify (counters *sycophantic agreement*: a tool that praises the
> boundary you wrote has reviewed your ego, not your evidence):**
> - [ ] Check each objection against your printed baseline and model scores. Reject
>   any that invents a number your output never produced.
> - [ ] Confirm it caught the causal reading if you left one in, and that your final
>   sentence names its uncertainty (the cross-validation spread, a subgroup gap, or a
>   shift boundary). Prediction answers *who*, never *why*.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

### 🗡️ Red-Team Exchange

The studio's **29–41 minute block**. Pair with one partner and run two rounds of
attack. Bring in an AI reviewer as a third voice using the **Prediction & Leakage
Auditor** role card (`ai_resources/agent_role_cards.md`; full briefing in
`genai_studio/roles/prediction_leakage_auditor.md`). This role is the **required**
reviewer for M7, whichever compass position you declared: a purely descriptive or
causal project runs it to confirm the protocol makes **no unlicensed prediction
claim**, and that confirmation is itself the ledgered result.

**Round 1: attack the baseline and the leakage trace (about 6 minutes).**
Your partner pastes your protocol to the auditor and reads back its **held-out and
baseline check** and its **leakage trace**. Is there a fair baseline to beat, or is a
score being reported with no dumb rule beside it? Was the held-out set truly held
out, or quietly peeked at? Is any feature on your list settled at or after the
outcome? For every leak they surface, you either trace and drop it in your own
pipeline or show why it does not apply. A flagged leak left in the protocol is an
open hole.

**Round 2: attack the claim boundary (about 6 minutes).**
Your partner hunts for overclaim in your one sentence: a metric with no baseline, a
boundary with no uncertainty, or the word *because* doing work your design has not
earned. The sharpest catch is a cause read off a feature weight. **Interpretability**
is noticing which features the model leaned on; **explanation** is knowing what
causes the outcome, and a leak can fake the first while telling you nothing about the
second. You defend the wording or rewrite it to the boundary your evidence actually
reaches.

**What to log.** Write down the single strongest hit from each round and how you
changed the protocol in response. An attack that changed nothing still gets a line
saying why it did not land. Those hits and your resolutions are the required M7
content of your ledger.

> **A question that often comes up here:** *"What if my partner and the auditor
> disagree about whether a feature leaks?"* Good. You are the judge, not either of
> them. Trace the feature's timing in your own data and say which case you found more
> convincing, in one sentence. And remember the correlated-error warning: if you also
> ask Gemini and it agrees "no leak," that is not confirmation. Two models can share
> the same blind spot. Agreement is not a clean bill; the timing in your own pipeline
> is.

### 📒 AI Research Ledger

Log every AI-assisted step in this sprint. One task per row. The **Decision** and
**Verification method** columns carry the weight: the output is a proposal, the
verification is what makes it usable. The auditor's leakage trace and your written
fixes are the required M7 ledger content. A missing ledger scores Craft 0 and returns
the submission (`ai_resources/ai_research_ledger_template.md`).

### YOUR ANSWER HERE:

| Task delegated | Tool used | Prompt | Output summary | Decision | Verification method | Remaining concern | Responsible researcher |
|---|---|---|---|---|---|---|---|
|  |  |  |  |  |  |  |  |
|  |  |  |  |  |  |  |  |

### ✅ Submission Checklist

Before you submit on Brightspace under **M07**, confirm:

- [ ] **Prediction verdict** written: the four-part contract in order (target,
  baseline, split, metric), **or** a defended non-prediction verdict naming the
  question your project really asks and its design.
- [ ] **Leakage trace** written: the one suspect feature, its timing check, and its
  correlation check; any feature that fails is dropped or re-timed.
- [ ] **Honesty checks + retract triggers** named, plus the **model-selection rule**
  fixed before any score and the criteria for calling the analysis **not useful**.
- [ ] **Claim boundary** written as one sentence carrying its uncertainty, with the
  forbidden crossing named: no causal reading of any weight.
- [ ] **Prediction & Leakage Auditor** run, its leakage trace and held-out/baseline
  check pasted, **every flag answered with your written fix**, and the **one
  recompute** named.
- [ ] **AI Research Ledger** filled, one row per AI-assisted step, every verification
  method named and non-vague.
- [ ] **3-minute walkthrough** ready; red-team notes logged (the strongest hit each
  round and what you changed).
- [ ] Everything traces to something real and reproducible. No dataset, feature, or
  number you cannot retrieve or rerun.

**Dossier update line.** Record in your Research Project Dossier that your declared-
analysis-protocol component now carries a prediction verdict (the contract or the
defended non-prediction verdict), a leakage trace, an honesty-check plan with retract
triggers, and a one-sentence claim boundary. Name the file or section where each now
lives. That line is your bridge into M8, where this declared protocol meets your data
for the first time and you run it end to end to produce first evidence.

<center>

Thank you!

</center>